# CSP 

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.model import Problem
from src.planning import Planner
from src.informed import search
from src.local_search import hill_climbing
from src.csp import solve_csp
from time import perf_counter
problem = Problem(ROOT / 'data_tiny')
t0 = perf_counter()
planner = Planner(problem)
preparation_seconds = perf_counter() - t0
print(f'Preparação comum: {preparation_seconds:.4f} s')
print('Lotes por onda:', [len(w) for w in planner.waves])
print('Valores por domínio:', [len(o) for o in planner.options])

Preparação comum: 0.0015 s
Lotes por onda: [3, 4, 3, 3, 4]
Valores por domínio: [18, 120, 24, 24, 120]


## Formulação e limites da pesquisa

Usamos o custo total em **quilómetros**. Cada linha de `order_items` é indivisível, mas linhas da mesma encomenda podem viajar separadamente. Não inventamos exigências de refrigeração, turnos ou limites de cais que não constam dos dados. A matriz fornecida determina distância e tempo; não usamos serviços externos.

Para tornar a comparação executável, restringimos o espaço de pesquisa a **cinco ondas**: recolhas de 30/10, entregas de 30/10, recolhas de 31/10, entregas de 31/10 recolhidas na véspera e restantes entregas de 31/10. Cada veículo executa as ondas por esta ordem, com relógio próprio; não precisa de esperar que o outro termine a onda. Itens do mesmo produtor/dia são agrupados; entregas são agrupadas por destino/dia e por recolha na véspera ou no próprio dia. A janela do lote é a interseção das janelas dos seus itens. Um lote não é dividido entre veículos. Se a interseção for vazia, o programa informa que esta formulação exige divisão do lote.

Uma decisão escolhe a distribuição dos lotes da onda pelos dois veículos e a ordem das visitas. Cada veículo faz no máximo uma viagem por onda, partindo e regressando ao armazém. Nas recolhas descarrega tudo ao regressar; nas entregas carrega antes de partir. Um veículo pode ficar parado numa onda. Estas escolhas são **hipóteses de implementação**, não exigências adicionais do enunciado.

O estado contém a onda seguinte, as duas rotas parciais, o instante de disponibilidade de cada veículo e os instantes de chegada e descarga de cada item no armazém. A disponibilidade para carregamento começa no fim da descarga. A permanência máxima de 24 h conta desde a chegada ao armazém até ao início do carregamento. Cada visita a produtor/cliente demora 10 min e cada carga/descarga demora 15 min. As janelas referem-se ao início do serviço. Esperas são permitidas; no carregamento esperamos pela partida útil para a primeira entrega.

Os sucessores enumeram todas as permutações e todos os cortes em duas sequências, rejeitando excessos de capacidade e violações temporais. O objetivo exige todos os itens entregues, regresso ao armazém e validação integral por `Problem.validate(require_all=True)`, incluindo inventário do armazém e precedências. A capacidade do armazém é verificada na solução completa, não usada como poda intermédia.

**Limite:** as conclusões de otimalidade de A* referem-se apenas a este espaço de ondas, lotes e horários determinísticos. Não se demonstra o ótimo do problema geral; falhar aqui também não prova que o problema geral é inviável. `data_small` permanece na EDA; esta comparação usa os dois veículos de `data_tiny`, como solicitado.

## CSP: variáveis, domínios e restrições

Cada variável `X_d` representa a decisão de uma onda. O seu domínio contém pares de sequências ordenadas de lotes, uma por veículo. Os domínios têm, neste conjunto, 18, 120, 24, 24 e 120 valores. A atribuição das cinco variáveis determina rotas e horários.

**Restrições duras:** cada lote é atribuído uma vez; capacidades dos veículos; janelas de recolha e entrega; encomenda anterior à recolha; fluxo produtor → armazém → cliente; carga após descarga; permanência de até 24 h; capacidade e inventário não negativo do armazém; horizonte e regresso ao depósito. A validação final confirma todas as restrições do modelo.

Usamos backtracking com ordem temporal fixa das variáveis, pois o relógio e stock de uma onda dependem das anteriores. O forward checking exige que a próxima onda conserve pelo menos um valor compatível; não faz propagação completa. Não afirmamos usar MRV. As variáveis das ondas têm restrições entre si, não são cinco problemas independentes.

A **preferência suave** é percorrer menos quilómetros: os valores são tentados por distância crescente. Isso orienta a pesquisa, mas não permite violar restrições duras. O CSP devolve a primeira atribuição viável, não uma prova de custo mínimo. Um backtrack é contado quando um valor localmente viável é abandonado por falta de continuação (incluindo poda pelo forward checking). Zero backtracks significa que o caminho escolhido não exigiu recuo, não que faltou implementar backtracking.

In [2]:
csp_result, csp_stats = solve_csp(planner)
assert csp_result.routes is not None, csp_result.status
validation = problem.validate(csp_result.routes)
assert validation.feasible, validation.errors
print('Custo:', round(csp_result.cost_km, 2), 'km')
print('Estatísticas:', csp_stats)
print(f'Tempo: {csp_result.seconds:.4f} s')
assignment = []
for d, option in enumerate(csp_result.choices):
    for vehicle_index, sequence in enumerate(option):
        assignment.append(dict(onda=d+1, veiculo=planner.vehicles[vehicle_index].id,
                              locais=[planner.waves[d][k].location for k in sequence],
                              itens=[i for k in sequence for i in planner.waves[d][k].items]))
display(pd.DataFrame(assignment))

Custo: 580.9 km
Estatísticas: {'nodes': 5, 'backtracks': 0, 'checks': 15}
Tempo: 0.0041 s


,onda,veiculo,locais,itens
0,1,1,[],[]
1,1,2,"[3, 2, 1]","[6, 11, 22, 24, 26, 29, 35, 12, 23, 27, 28, 30..."
2,2,1,[],[]
3,2,2,"[5, 7, 8, 6]","[32, 33, 34, 35, 11, 12, 13, 30, 31, 36, 37, 3..."
4,3,1,[],[]
5,3,2,"[3, 2, 1]","[3, 4, 8, 9, 14, 17, 1, 7, 15, 16, 18, 2, 10, ..."
6,4,1,"[9, 8, 6]","[25, 26, 27, 28, 29, 5, 6, 22, 23, 24]"
7,4,2,[],[]
8,5,1,[],[]
9,5,2,"[5, 7, 9, 6]","[14, 15, 1, 2, 3, 4, 16, 17, 18, 19, 20, 21, 7..."
